In [1]:
import os
import numpy as np
import pandas as pd
import cv2
import random
from pyts.image import GramianAngularField
from joblib import Parallel, delayed

import matplotlib.pyplot as plt
from pyts.image import RecurrencePlot

from pyts.image import MarkovTransitionField
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import scipy.signal
import concurrent.futures


from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import TSNE


import pickle

from PIL import Image

## GAF

In [ ]:
selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

df = pd.read_csv('data/csv/cicddos_2019_6_labels.csv')
df = df[selected_columns]


len(selected_columns)

def apply_augmentations(image):
    # Random blur
    if random.random() > 0.5:
        image = cv2.GaussianBlur(image, (5, 5), 0)
    
    # Add random noise
    if random.random() > 0.5:
        noise = np.random.normal(0, 10, image.shape).astype(np.uint8)
        image = cv2.add(image, noise)
    
    # Random flip
    if random.random() > 0.5:
        image = cv2.flip(image, 1)
    
    # Random rotation
    angle = random.choice([0, 90, 180, 270])
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
    image = cv2.warpAffine(image, matrix, (w, h))
    
    return image

def get_next_filename(label_dir):
    existing_files = sorted([int(f.split('.')[0]) for f in os.listdir(label_dir) if f.endswith('.png')])
    return existing_files[-1] + 1 if existing_files else 1

def upscale_image(image):
    num_features = 49
    size = 224
    upscale_factor = size // num_features
    return np.repeat(np.repeat(image, upscale_factor, axis=0), upscale_factor, axis=1)

def convert_to_gaf_and_save(row, label, output_dir):
    gaf = GramianAngularField(method='summation')
    
    # Chuyển đổi dữ liệu thành ma trận GAF
    data = np.array(row, dtype=np.float32).reshape(1, -1)
    image = gaf.fit_transform(data)[0]
    
    # Chuẩn hóa dữ liệu ảnh về dạng uint8
    image = ((image - np.min(image)) / (np.max(image) - np.min(image)) * 255).astype(np.uint8)
    
    # Resize về 224x224
    image = upscale_image(image)
    image = cv2.resize(image, (224, 224))

    # Áp dụng các biến đổi ảnh
    # image = apply_augmentations(image)
    
    # Lưu ảnh
    label_dir = os.path.join(output_dir, str(label))
    os.makedirs(label_dir, exist_ok=True)
    
    # Giới hạn số ảnh trong mỗi nhãn là 200
    if len(os.listdir(label_dir)) >= 200:
        return
    
    filename = os.path.join(label_dir, f"{get_next_filename(label_dir)}.png")
    cv2.imwrite(filename, image)

def process_dataframe(output_dir):
    labels = df['Label']  # Cột cuối là nhãn
    features = df.drop(columns=['Label'])  # Các cột còn lại là dữ liệu

    features.replace([-np.inf, np.inf], 0, inplace=True)
    features.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị


    std = features.std()
    std.replace(0, 1, inplace=True)

    features = np.log1p(features + 1)
    features.replace([-np.inf, np.inf], 0, inplace=True)
    features.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    # scaler = MinMaxScaler(feature_range=(0, 255))
    # features_scaled = scaler.fit_transform(features).astype(np.uint8)

    # # Tạo DataFrame mới nhưng giữ lại index và column names
    # features = pd.DataFrame(features_scaled, index=features.index, columns=features.columns)
    
    Parallel(n_jobs=3)(
        delayed(convert_to_gaf_and_save)(features.iloc[i].values, labels.iloc[i], output_dir)
        for i in range(len(features))
    )

if __name__ == "__main__":
    # Đọc dữ liệu từ file CSV
    output_directory = "temp/test_chuyen_doi_anh/GAF"
    
    # Xử lý và lưu ảnh
    process_dataframe(output_directory)
    print("Xử lý hoàn tất!")


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


Xử lý hoàn tất!


## Recurrence Plot (RP)

In [10]:
def upscale_image(image):
    num_features = 49
    size = 224
    upscale_factor = size // num_features
    return np.repeat(np.repeat(image, upscale_factor, axis=0), upscale_factor, axis=1)

def apply_augmentations(image):
    # Random blur
    if random.random() > 0.5:
        image = cv2.GaussianBlur(image, (5, 5), 0)
    
    # Add random noise
    if random.random() > 0.5:
        noise = np.random.normal(0, 10, image.shape).astype(np.uint8)
        image = cv2.add(image, noise)
    
    # Random flip
    if random.random() > 0.5:
        image = cv2.flip(image, 1)
    
    # Random rotation
    angle = random.choice([0, 90, 180, 270])
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
    image = cv2.warpAffine(image, matrix, (w, h))
    
    return image

# Định nghĩa hàm vẽ và lưu Recurrence Plot
def save_recurrence_plot(data, label, output_dir, index):
    label_dir = os.path.join(output_dir, str(label))
    os.makedirs(label_dir, exist_ok=True)
    
    file_path = os.path.join(label_dir, f"rp_{index}.png")
    if os.path.exists(file_path):
        return  # Bỏ qua nếu file đã tồn tại
    
    # Chuyển đổi dữ liệu thành ảnh Recurrence Plot
    rp = RecurrencePlot()
    image = rp.fit_transform([data])[0]

    # Chuẩn hóa dữ liệu ảnh về dạng uint8
    image = ((image - np.min(image)) / (np.max(image) - np.min(image)) * 255).astype(np.uint8)
    
    # Resize về 224x224
    image = upscale_image(image)
    image = cv2.resize(image, (224, 224))

    # Áp dụng các biến đổi ảnh
    # image = apply_augmentations(image)
    
    # Lưu ảnh
    plt.figure(figsize=(2, 2))  # Giữ ảnh nhỏ để tiết kiệm bộ nhớ
    plt.imshow(image, cmap='gray', origin='lower')
    plt.axis('off')
    plt.savefig(file_path, bbox_inches='tight', pad_inches=0)
    plt.close()

# Đọc dữ liệu
def process_dataset(file_path, output_dir, max_images_per_label=200, n_jobs=3):
    df = pd.read_csv(file_path)
    labels = df['Label']  # Cột cuối là nhãn
    features = df.drop(columns=['Label'])  # Các cột còn lại là dữ liệu

    features.replace([-np.inf, np.inf], 0, inplace=True)
    features.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
    std = features.std()
    std.replace(0, 1, inplace=True)
    features = np.log1p(features + 1)
    features.replace([-np.inf, np.inf], 0, inplace=True)
    features.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    data = features.values  # Các cột trước là dữ liệu
    
    unique_labels = np.unique(labels)
    tasks = []
    
    for label in unique_labels:
        label_indices = np.where(labels == label)[0][:max_images_per_label]  # Chỉ lấy tối đa 200 ảnh mỗi nhãn
        for idx in label_indices:
            tasks.append((data[idx], label, output_dir, idx))
    
    # Chạy song song với 3 luồng xử lý
    Parallel(n_jobs=n_jobs)(delayed(save_recurrence_plot)(*task) for task in tasks)

# Chạy chương trình
input_csv = "data/csv/cicddos_2019_6_labels.csv"  # Thay bằng đường dẫn thực tế
output_folder = "temp/test_chuyen_doi_anh/Recurrence_Plot"
process_dataset(input_csv, output_folder)

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


## Markov Transition Field

In [ ]:
def upscale_image(image):
    num_features = 49
    size = 224
    upscale_factor = size // num_features
    return np.repeat(np.repeat(image, upscale_factor, axis=0), upscale_factor, axis=1)

def apply_augmentations(image):
    # Random blur
    if random.random() > 0.5:
        image = cv2.GaussianBlur(image, (5, 5), 0)
    
    # Add random noise
    if random.random() > 0.5:
        noise = np.random.normal(0, 10, image.shape).astype(np.uint8)
        image = cv2.add(image, noise)
    
    # Random flip
    if random.random() > 0.5:
        image = cv2.flip(image, 1)
    
    # Random rotation
    angle = random.choice([0, 90, 180, 270])
    h, w = image.shape[:2]
    matrix = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1)
    image = cv2.warpAffine(image, matrix, (w, h))
    
    return image

def save_mtf_image(data_row, label, img_index, save_dir):
    try:
        # Định nghĩa MTF
        mtf = MarkovTransitionField(image_size=32)  # Kích thước ảnh 32x32
        data_row = data_row.reshape(1, -1)  # Định dạng lại dữ liệu
        image = mtf.fit_transform(data_row)[0]

        # Chuẩn hóa dữ liệu ảnh về dạng uint8
        image = ((image - np.min(image)) / (np.max(image) - np.min(image)) * 255).astype(np.uint8)
        
        # Resize về 224x224
        image = upscale_image(image)
        image = cv2.resize(image, (224, 224))

        # Áp dụng các biến đổi ảnh
        # image = apply_augmentations(image)
        
        # Tạo thư mục lưu ảnh
        label_dir = os.path.join(save_dir, str(label))
        os.makedirs(label_dir, exist_ok=True)
        
        # Lưu ảnh
        img_path = os.path.join(label_dir, f"{img_index}.png")
        plt.imsave(img_path, image, cmap='gray')
    except Exception as e:
        print(f"Lỗi khi xử lý ảnh {img_index} của nhãn {label}: {e}")

def process_label_group(group, label, save_dir):
    max_images = 200  # Giới hạn số ảnh lưu
    group = group[:max_images]  # Cắt bớt nếu cần
    with ThreadPoolExecutor(max_workers=3) as executor:
        for img_index, row in enumerate(group.values):
            executor.submit(save_mtf_image, row, label, img_index, save_dir)

def main():
    # Đọc dữ liệu
    data_path = "data/csv/cicddos_2019_6_labels.csv"  # Thay đổi đường dẫn tùy vào dữ liệu của bạn
    save_dir = "temp/test_chuyen_doi_anh/Markov Transition Field"
    df = pd.read_csv(data_path)
    
    
    y = df['Label']  # Cột cuối là nhãn
    features = df.drop(columns=['Label'])  # Các cột còn lại là dữ liệu

    features.replace([-np.inf, np.inf], 0, inplace=True)
    features.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
    std = features.std()
    std.replace(0, 1, inplace=True)
    features = np.log1p(features + 1)
    features.replace([-np.inf, np.inf], 0, inplace=True)
    features.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    X = features

    # Nhóm dữ liệu theo nhãn
    grouped = X.groupby(y)
    
    with ThreadPoolExecutor(max_workers=3) as executor:
        for label, group in grouped:
            executor.submit(process_label_group, group, label, save_dir)

main()

## Spectrogram (STFT - Wavelet)

In [ ]:
# Định nghĩa các tham số
N_FFT = 256  # Số điểm FFT
HOP_LENGTH = 128  # Bước nhảy
OUTPUT_DIR = "temp/test_chuyen_doi_anh/Spectrogram (STFT - Wavelet)"
MAX_IMAGES_PER_LABEL = 200
NUM_THREADS = 3


def normalize_data(data):
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
    std = data.std()
    std.replace(0, 1, inplace=True)
    data = np.log1p(data + 1)
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
    return data

def generate_spectrogram(data, output_path, label, index):
    os.makedirs(output_path, exist_ok=True)
    
    f, t, Zxx = scipy.signal.stft(data, nperseg=128)
    Zxx = np.abs(Zxx)
    
    fig, ax = plt.subplots(figsize=(2, 2))  # Small image size
    ax.pcolormesh(t, f, Zxx, shading='gouraud')
    ax.axis('off')
    
    img_path = os.path.join(output_path, f"{label}_{index}.png")

    pixel_scale = 3 # giảm kích thước ảnh 3 lần
    dpi = 300 / pixel_scale

    fig.savefig(img_path, bbox_inches='tight', pad_inches=0, dpi=dpi)
    plt.close(fig)

def process_label(grouped_data, label, output_dir, max_images=200):
    output_path = os.path.join(output_dir, str(label))
    os.makedirs(output_path, exist_ok=True)
    
    for idx, (_, row) in enumerate(grouped_data.iterrows()):
        if idx >= max_images:
            break
        generate_spectrogram(row.values, output_path, label, idx)

def main(input_csv, output_dir):
    df = pd.read_csv(input_csv)
    labels = df['Label'].unique()
    
    os.makedirs(output_dir, exist_ok=True)
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = []
        for label in labels:
            label_data = df[df['Label'] == label].drop(columns=['Label'])
            normalized_data = normalize_data(label_data)
            futures.append(executor.submit(process_label, pd.DataFrame(normalized_data), label, output_dir))
        
        concurrent.futures.wait(futures)

input_csv = "data/csv/cicddos_2019_6_labels.csv"  # Thay bằng file dữ liệu của bạn
output_dir = "temp/test_chuyen_doi_anh/Spectrogram (STFT - Wavelet)"
main(input_csv, output_dir)


f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\scipy\signal\_spectral_py.py:1430: UserWarning: nperseg = 128 is greater than input length  = 77, using nperseg = 77
  freqs, time, Zxx = _spectral_helper(x, x, fs, window, nperseg, noverlap,
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid

## DeepInsight

In [2]:
selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']

def process_task(task):
    row, label, img_number, output_base = task
    num_features = row.shape[0]
    s = int(np.ceil(np.sqrt(num_features)))
    padded_size = s * s
    padded_row = np.zeros(padded_size)
    padded_row[:num_features] = row
    image_2d = padded_row.reshape((s, s))

    size = 224
    upscale_factor= size // s 
    expanded_image = np.kron(image_2d, np.ones((upscale_factor, upscale_factor)))

    scaled_image = (expanded_image * 255).astype(np.uint8)

    img = Image.fromarray(scaled_image, mode='L')
    img = img.convert("RGB")
    img = img.resize((size, size))
    img.save(os.path.join(output_base, str(label), f'{img_number}.png'))

def deepinsight_conversion(input_csv, output_dir='output_images', max_per_label=200, workers=3):
    # Đọc dữ liệu
    df = pd.read_csv(input_csv)
    df = df[selected_columns]

    labels = df['Label']
    data = df.drop(columns=['Label'])

    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
    std = data.std()
    std.replace(0, 1, inplace=True)
    data = np.log1p(data + 1)
    data.replace([-np.inf, np.inf], 0, inplace=True)
    data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

    features = data.values

    # Chuẩn hóa dữ liệu
    scaler = MinMaxScaler()
    normalized_features = scaler.fit_transform(features)
    # normalized_features = features

    # Tạo thư mục đầu ra
    os.makedirs(output_dir, exist_ok=True)
    unique_labels = pd.unique(labels)
    for label in unique_labels:
        os.makedirs(os.path.join(output_dir, str(label)), exist_ok=True)

    # Chuẩn bị tasks
    tasks = []
    for label in unique_labels:
        mask = (labels == label).values
        label_features = normalized_features[mask][:max_per_label]
        for img_number, row in enumerate(label_features):
            tasks.append((row, label, img_number, output_dir))

    # Xử lý song song
    with ThreadPoolExecutor(max_workers=workers) as executor:
        executor.map(process_task, tasks)

# Sử dụng hàm
deepinsight_conversion(
    input_csv='data/csv/cicddos_2019_6_labels.csv',
    output_dir='temp/test_chuyen_doi_anh/DeepInsight',
    max_per_label=200,
    workers=3
)

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


In [38]:
selected_columns = ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                    'Fwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 
                    'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 
                    'Bwd Packet Length Mean', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
                    'Flow IAT Max', 'Flow IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Min', 
                    'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 
                    'Bwd Header Length', 'Bwd Packets/s', 'Packet Length Max', 'FIN Flag Count', 'SYN Flag Count', 
                    'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 
                    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
                    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Init Bwd Win Bytes', 
                    'Fwd Seg Size Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Std', 'Label']


input_csv='data/csv/cicddos_2019_6_labels.csv'
df = pd.read_csv(input_csv)
df = df[selected_columns]

data = df.drop(columns=['Label'])
data.replace([-np.inf, np.inf], 0, inplace=True)
data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị
std = data.std()
std.replace(0, 1, inplace=True)
data = np.log1p(data + 1)
data.replace([-np.inf, np.inf], 0, inplace=True)
data.fillna(0, inplace=True)  # Điền giá trị NaN bằng giá trị trung vị

train_data = data.values

# Giả sử train_data là dữ liệu training ban đầu
scaler = MinMaxScaler()

scaler.fit(train_data)  # Fit scaler với toàn bộ tập huấn luyện

# Lưu scaler đã fit vào file để dùng sau này
with open("minmax_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: divide by zero encountered in log1p
  result = func(self.values, **kwargs)
f:\NCKH\code\DN\IDS-RESNET-50-System\train_model\.venv\Lib\site-packages\pandas\core\internals\blocks.py:393: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)
